# 📊 Tesis Evaluation: Ground Truth Perbandingan (Compare Systems)
Notebook ini menghitung dan membandingkan **MRR (Mean Reciprocal Rank)** antara **Legal Tina** dan **Vector Search (Qdrant)**.

**Definisi MRR di sini:** rata-rata $1/\text{rank}$ dari seluruh dokumen yang di-rank oleh sistem untuk kueri tersebut, lalu dirata-rata antar kueri. Kolom *hit rate* adalah bukti IR bahwa kedua sistem berhasil me-retrieve.

In [14]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.colheader_justify', 'center')

df = pd.read_csv("Ground truth - MRR COMPARE (2).csv")
print(f"✅ Loaded: {len(df)} baris, {df['query_id'].nunique()} kueri")

✅ Loaded: 30 baris, 3 kueri


### 🔍 Pratinjau Data (10 baris pertama)

In [15]:
cols_show = ['query_id', 'query', 'agreement_title', 'rank_legal_tina', 'rank_qdrant', 'hit rate']
df[cols_show].head(10)

,query_id,query,agreement_title,rank_legal_tina,rank_qdrant,hit rate
0,Q01,rules of origin,EU - Chile Interim Trade Agreement - 2025,5.0,NaN,1
1,Q01,rules of origin,China - Maldives - 2025,3.0,NaN,1
2,Q01,rules of origin,EAEU - New Zealand - 2025,2.0,NaN,1
3,Q01,rules of origin,United Arab Emirates - Malaysia CEPA - 2025,2.0,NaN,1
4,Q01,rules of origin,United Arab Emirates - Chile CEPA - 2025,2.0,NaN,1
5,Q01,rules of origin,Japan - Switzerland,NaN,2.0,1
6,Q01,rules of origin,"ASEAN - Korea, Republic of",NaN,2.0,1
7,Q01,rules of origin,United Kingdom - European Union,NaN,4.0,1
8,Q01,rules of origin,"China - Macao, China",NaN,1.0,1
9,Q01,rules of origin,"China - Hong Kong, China",NaN,1.0,1


### 🛠️ Hitung RR per Baris
$RR = 1/\text{rank}$ untuk setiap rank (rank 0 → RR 0). Lalu agregasi per kueri.

In [16]:
# RR per baris: 1/rank, rank 0 atau NaN → 0
df['rr_tina']   = df['rank_legal_tina'].apply(lambda r: 1/r if pd.notna(r) and r > 0 else 0)
df['rr_qdrant'] = df['rank_qdrant'].apply(lambda r: 1/r if pd.notna(r) and r > 0 else 0)

# Per-query summary
summary = df.groupby('query_id').agg(
    query_text       = ('query', 'first'),
    n_tina_ranked    = ('rank_legal_tina', lambda x: x.notna().sum()),
    n_qdrant_ranked  = ('rank_qdrant', lambda x: x.notna().sum()),
    mean_rank_tina   = ('rank_legal_tina', 'mean'),
    mean_rank_qdrant = ('rank_qdrant', 'mean'),
    mrr_tina         = ('rr_tina', lambda x: x[df.loc[x.index, 'rank_legal_tina'].notna()].mean()),
    mrr_qdrant       = ('rr_qdrant', lambda x: x[df.loc[x.index, 'rank_qdrant'].notna()].mean()),
    hit_rate_tina    = ('rank_legal_tina', lambda x: (x.dropna() > 0).mean()),
    hit_rate_qdrant  = ('rank_qdrant', lambda x: (x.dropna() > 0).mean()),
)

summary

,query_text,n_tina_ranked,n_qdrant_ranked,mean_rank_tina,mean_rank_qdrant,mrr_tina,mrr_qdrant,hit_rate_tina,hit_rate_qdrant
query_id,,,,,,,,,
Q01,rules of origin,5,5,2.8,2.0,0.406667,0.65,1.0,1.0
Q02,certificate of origin,5,5,1.6,0.8,0.316667,0.80,0.6,0.8
Q03,technical barriers to trade,5,5,2.2,1.8,0.400000,0.75,0.8,1.0


### 📋 Detail per Kueri: Daftar Rank & RR masing-masing Sistem

In [17]:
for qid in df['query_id'].unique():
    sub = df[df['query_id'] == qid]
    q_text = sub['query'].iloc[0]

    tina_rr  = sub[sub['rank_legal_tina'].notna()][['rank_legal_tina', 'rr_tina']].values
    qdr_rr   = sub[sub['rank_qdrant'].notna()][['rank_qdrant', 'rr_qdrant']].values

    print(f"\n▶️  [{qid}] \"{q_text}\"")
    print(f"   Legal Tina   ranks: {[int(r) for r,_ in tina_rr]}")
    print(f"                RRs  : {[round(rr,4) for _,rr in tina_rr]} → MRR = {np.mean([rr for _,rr in tina_rr]):.4f}")
    print(f"   Vector Search ranks: {[int(r) for r,_ in qdr_rr]}")
    print(f"                RRs  : {[round(rr,4) for _,rr in qdr_rr]} → MRR = {np.mean([rr for _,rr in qdr_rr]):.4f}")


▶️  [Q01] "rules of origin"
   Legal Tina   ranks: [5, 3, 2, 2, 2]
                RRs  : [np.float64(0.2), np.float64(0.3333), np.float64(0.5), np.float64(0.5), np.float64(0.5)] → MRR = 0.4067
   Vector Search ranks: [2, 2, 4, 1, 1]
                RRs  : [np.float64(0.5), np.float64(0.5), np.float64(0.25), np.float64(1.0), np.float64(1.0)] → MRR = 0.6500

▶️  [Q02] "certificate of origin"
   Legal Tina   ranks: [0, 1, 3, 4, 0]
                RRs  : [np.float64(0.0), np.float64(1.0), np.float64(0.3333), np.float64(0.25), np.float64(0.0)] → MRR = 0.3167
   Vector Search ranks: [0, 1, 1, 1, 1]
                RRs  : [np.float64(0.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0)] → MRR = 0.8000

▶️  [Q03] "technical barriers to trade"
   Legal Tina   ranks: [1, 2, 4, 0, 4]
                RRs  : [np.float64(1.0), np.float64(0.5), np.float64(0.25), np.float64(0.0), np.float64(0.25)] → MRR = 0.4000
   Vector Search ranks: [4, 1, 2, 1, 1]
                RRs  : [np.f

### 🏁 Rekapitulasi Akhir

In [18]:
mrr_tina   = summary['mrr_tina'].mean()
mrr_qdrant = summary['mrr_qdrant'].mean()
rank_tina  = summary['mean_rank_tina'].mean()
rank_qdrant = summary['mean_rank_qdrant'].mean()
improve    = (mrr_qdrant - mrr_tina) / mrr_tina * 100

print("┌" + "─" * 72 + "┐")
print(f"│ {'HEAD-TO-HEAD MRR EVALUATION'.center(70)} │")
print("├" + "─" * 72 + "┤")
print(f"│ {'Metrik':<30} {'Legal Tina':>18} {'Vector Search':>18} │")
print("├" + "─" * 72 + "┤")

for _, row in summary.iterrows():
    print(f"│ {row['query_text'][:28]:<28}  MRR {row['mrr_tina']:>14.4f}  {row['mrr_qdrant']:>14.4f} │")

print("├" + "─" * 72 + "┤")
print(f"│ {'🏆 MRR (rata-rata)':<30} {mrr_tina:>18.4f} {mrr_qdrant:>18.4f} │")
print(f"│ {'🏆 Mean Rank (rata-rata)':<30} {rank_tina:>18.4f} {rank_qdrant:>18.4f} │")
print(f"│ {'📈 Improvement MRR':<30} {'':>18} {improve:>17.2f}% │")
print("└" + "─" * 72 + "┘")

┌────────────────────────────────────────────────────────────────────────┐
│                      HEAD-TO-HEAD MRR EVALUATION                       │
├────────────────────────────────────────────────────────────────────────┤
│ Metrik                                 Legal Tina      Vector Search │
├────────────────────────────────────────────────────────────────────────┤
│ rules of origin               MRR         0.4067          0.6500 │
│ certificate of origin         MRR         0.3167          0.8000 │
│ technical barriers to trade   MRR         0.4000          0.7500 │
├────────────────────────────────────────────────────────────────────────┤
│ 🏆 MRR (rata-rata)                          0.3744             0.7333 │
│ 🏆 Mean Rank (rata-rata)                    2.2000             1.5333 │
│ 📈 Improvement MRR                                             95.85% │
└────────────────────────────────────────────────────────────────────────┘
